In [1]:
import pandas as pd
df = pd.read_csv(r"C:\Users\chhay\Downloads\healthcare_data_cleaning_dataset.csv")
df

,Patient_ID,Age,Gender,City,Diagnosis,Hospital_Visits,Treatment_Cost,Insurance_Coverage,Admission_Date
0,17270,35.0,Male,Bangalore,Hypertension,13,41010.0,1,2023-11-30
1,10860,21.0,Female,Hyderabad,Flu,11,12194.0,1,2023-02-23
2,15390,77.0,Female,Bangalore,Asthma,2,45086.0,0,2023-03-14
3,15191,79.0,Female,Mumbai,Asthma,13,40842.0,0,2023-08-01
4,15734,60.0,Female,Delhi,Asthma,1,9873.0,1,2023-06-20
...,...,...,...,...,...,...,...,...,...
5095,11764,NaN,Female,Mumbai,COVID-19,15,NaN,0,2023-09-22
5096,17597,NaN,Female,Chennai,Asthma,2,NaN,0,2023-06-26
5097,19171,NaN,Female,Mumbai,Flu,1,NaN,1,2023-12-31
5098,13854,NaN,Female,Bangalore,Flu,17,NaN,0,2023-01-18


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5100 entries, 0 to 5099
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Patient_ID          5100 non-null   int64  
 1   Age                 4500 non-null   float64
 2   Gender              5100 non-null   object 
 3   City                5100 non-null   object 
 4   Diagnosis           5100 non-null   object 
 5   Hospital_Visits     5100 non-null   int64  
 6   Treatment_Cost      4507 non-null   float64
 7   Insurance_Coverage  5100 non-null   int64  
 8   Admission_Date      5100 non-null   object 
dtypes: float64(2), int64(3), object(4)
memory usage: 358.7+ KB


In [5]:
df.describe()

,Patient_ID,Age,Hospital_Visits,Treatment_Cost,Insurance_Coverage
count,5100.000000,4500.000000,5100.000000,4507.000000,5100.000000
mean,14967.003529,49.597556,9.981765,26920.151157,0.489804
std,2869.152551,28.663852,5.464559,23224.930985,0.499945
min,10001.000000,0.000000,1.000000,526.000000,0.000000
25%,12479.000000,25.000000,5.000000,12498.000000,0.000000
50%,14976.000000,50.000000,10.000000,24797.000000,0.000000
75%,17386.000000,74.000000,15.000000,37922.000000,1.000000
max,19998.000000,99.000000,19.000000,199702.965333,1.000000


## Q1. Missing Data Identification

Scenario: 

The hospital suspects incomplete patient records.

Task:

* Identify missing values in each column
* Calculate percentage of missing data

In [18]:
df.isnull().sum()

Patient_ID              0
Age                     0
Gender                  0
City                    0
Diagnosis               0
Hospital_Visits         0
Treatment_Cost        593
Insurance_Coverage      0
Admission_Date          0
dtype: int64

In [12]:
df.isnull().mean()*100

Patient_ID             0.000000
Age                   11.764706
Gender                 0.000000
City                   0.000000
Diagnosis              0.000000
Hospital_Visits        0.000000
Treatment_Cost        11.627451
Insurance_Coverage     0.000000
Admission_Date         0.000000
dtype: float64

## Q2. Handling Missing Age

Scenario:

Age is critical for medical analysis, but some values are missing.
    
Task:

* Replace missing Age values with an appropriate method

* Justify your choice (mean/median    :

  ## I selected median to fill the Age missing values because here min age is 0 & max age is 99 and median will calculate on the basis of position not on the values calculation.

In [17]:
age_median = df['Age'].median()
    
df['Age'] = df['Age'].fillna(age_median)

print(df['Age'].isnull().sum())

0


## Q3. Handling Missing Treatment Cost

Scenario:

Treatment cost is highly skewed due to expensive treatments.
    
Task:

* Handle missing Treatment_Cost values

* Choose the correct imputation method and explain why

  ### I prefered median here to handle this missing value , as I checked above the 75% data under 38K while max treatment_cost is ~ 199702 which seems to be a abnormal hence mean approach will fail.


In [28]:
cost_median = df['Treatment_Cost'].median()

df['Treatment_Cost'] = df['Treatment_Cost'].fillna(cost_median)

print(df['Treatment_Cost'].isnull().sum())


0


## Q4. Duplicate Patient Records

Scenario: 

Some patient records were entered multiple times.
    
Task:
    
* Identify duplicate rows

* Remove duplicates

* Compare dataset size before and after

In [35]:
df.duplicated().sum()
df = df.drop_duplicates()

print(df)

,Patient_ID,Age,Gender,City,Diagnosis,Hospital_Visits,Treatment_Cost,Insurance_Coverage,Admission_Date
0,17270,35.0,Male,Bangalore,Hypertension,13,41010.0,1,2023-11-30
1,10860,21.0,Female,Hyderabad,Flu,11,12194.0,1,2023-02-23
2,15390,77.0,Female,Bangalore,Asthma,2,45086.0,0,2023-03-14
3,15191,79.0,Female,Mumbai,Asthma,13,40842.0,0,2023-08-01
4,15734,60.0,Female,Delhi,Asthma,1,9873.0,1,2023-06-20
...,...,...,...,...,...,...,...,...,...
4996,16135,50.0,Female,Delhi,Hypertension,13,24797.0,1,2023-11-02
4997,15573,35.0,Female,Delhi,COVID-19,19,2585.0,1,2023-12-02
4998,14131,60.0,Female,Chennai,Hypertension,4,31337.0,0,2023-04-07
4999,19900,16.0,Female,Chennai,Flu,16,39801.0,0,2023-08-04


## Q5. Invalid Age Values (Data Quality Check)

Scenario: 

Some patients have unrealistic age values (e.g., >100 or <0).

Task:

* Detect such records

* Decide whether to remove or correct them

In [3]:
unrealistic_age = ((df['Age'] < 0) | (df['Age'] > 100)).sum()
print(unrealistic_age)

0


### No such values identified 

##  Q6. Outlier Detection (Treatment Cost)

Scenario: 

Extreme treatment costs are affecting analysis.

Task:

* Detect outliers using IQR method
* Display number of outliers


In [5]:
Q1 = df['Treatment_Cost'].quantile(0.25)
Q3 = df['Treatment_Cost'].quantile(0.75)
IQR = Q3-Q1

lower_bound = Q1-1.5*IQR
upper_bound = Q3+1.5*IQR

print(lower_bound)
print(upper_bound)


-25638.0
76058.0


In [8]:
Total_outliers = ((df['Treatment_Cost'] > upper_bound) | (df['Treatment_Cost'] < lower_bound)).sum()
print(f"Total_outliers(High+Low): {Total_outliers}")

Total_outliers(High+Low): 50


### Q7. Outlier Treatment

Scenario: 

The business team wants to retain all records.

Task:

* Apply capping (Winsorization) on Treatment_Cost

* Use 5th and 95th percentile

In [15]:
lower_limit = df['Treatment_Cost'].quantile(.05)
upper_limit = df['Treatment_Cost'].quantile(0.95)

print(f"lower_limit(5th): {lower_limit}")
print(f"upper_limit(95th): {upper_limit}")

df['Treatment_Cost'] = df['Treatment_Cost'].clip(lower=lower_limit, upper=upper_limit)

lower_limit(5th): 2914.6
upper_limit(95th): 48188.1


### Q8. Transformation

Scenario: 

Treatment cost is highly skewed.
    
Task:

* Apply log transformation

* Create a new column

* Compare before vs after distribution

NA

##  Q9. Time-Based Missing Handling

Scenario: 

Admission dates should follow a logical sequence.

Task:

* Sort data by Admission_Date
* Apply forward fill or backward fill where appropriate
* Justify your choice

In [5]:
df = df.sort_values(by = 'Admission_Date').reset_index(drop = True)
df['Admission_Date_cleaned'] = df['Admission_Date'].ffill()

### I used forward filling to update date the reason to stop data leakage , here we take NAn date from the previous date if we use 
backward fill data consitancy depened on future date which is not good in given data set